# 00 · How we use the RAG API

**Official Walkthrough · Dingren Daoxue Lab · CC0 1.0**

This notebook is the minimum example: **30 lines of Python** to query the public Daoism RAG API at [lius.cc/api/llm-rag](https://lius.cc/api/llm-rag) and feed results to any LLM.

- 📦 Repo: https://github.com/lius-cc/lius-cookbook
- 🤖 Model: [lius-cc/Daoism-Qwen3.5-9B](https://huggingface.co/lius-cc/Daoism-Qwen3.5-9B)
- 📚 Dataset: [lius-cc/daoism-knowledge-rag](https://huggingface.co/datasets/lius-cc/daoism-knowledge-rag) (95,919 nodes)
- 🌐 Live demos: https://lius.cc/llm/demos
- 📜 v0.1 preprint — not yet peer-reviewed.

## Open in Colab
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lius-cc/lius-cookbook/blob/main/notebooks/how-we-use-rag-api.ipynb)

## 1 · Bare-minimum call

No deps beyond `requests`.

In [ ]:
import requests

RAG_API = "https://lius.cc/api/llm-rag"  # 30 req/min · free

r = requests.post(RAG_API, json={"q": "三朝醮", "n": 5}, timeout=10).json()

for hit in r["hits"]:
    print(f"[{hit['type']:9}] {hit['name']:30}  score={hit['score']}  {hit['url']}")

## 2 · Feed RAG results into an LLM

Works with any OpenAI-compatible endpoint (vLLM, llama-cpp-python, Ollama with the openai compat shim, or hosted models). Below uses a local vLLM at `:8000`.

In [ ]:
import requests
from openai import OpenAI

client = OpenAI(api_key="dummy", base_url="http://localhost:8000/v1")

def ask(question: str):
    hits = requests.post(
        "https://lius.cc/api/llm-rag",
        json={"q": question, "n": 5},
        timeout=10,
    ).json()["hits"]

    ctx = "\n\n---\n\n".join(
        f"### {h['name']}\n{h['summary']}\n\n{h['content'][:1500]}"
        for h in hits
    )

    return client.chat.completions.create(
        model="lius-cc/Daoism-Qwen3.5-9B",
        messages=[
            {"role": "system", "content": f"Answer based on these canonical entries:\n\n{ctx}"},
            {"role": "user", "content": question},
        ],
        max_tokens=4096,
    ).choices[0].message.content

# print(ask("請說明三朝醮的完整流程"))

## 3 · Reproducibility lock

If you cite this notebook in a paper, lock these:

In [ ]:
REPRO = {
    "model":   "lius-cc/Daoism-Qwen3.5-9B",  # Apache 2.0
    "dataset": "lius-cc/daoism-knowledge-rag@v1",  # 95,919 nodes
    "rag_api": "https://lius.cc/api/llm-rag",  # 30 req/min, free
    "snapshot": "2026-05-17",
    "max_tokens": 4096,
    "temperature": 0.0,
}
REPRO

## 4 · Cite us

```bibtex
@misc{daoism-qwen3-9b-2026,
  author = {Liu, Chi-Ying and Dingren Daoxue Lab},
  title  = {Daoism-Qwen3.5-9B: An Open-Source Taoist Knowledge Language Model},
  year   = {2026},
  publisher = {HuggingFace},
  url    = {https://huggingface.co/lius-cc/Daoism-Qwen3.5-9B}
}
```

**v0.1 preprint** — citable via Zenodo DOI [10.5281/zenodo.20248697](https://doi.org/10.5281/zenodo.20248697); formal peer-reviewed version pending journal submission.